# Proyecto 4: Asistente de análisis de datos

Agente que analiza datasets CSV con lenguaje natural usando tool use de Claude.
El usuario describe lo que quiere ver; el agente ejecuta Pandas de forma segura.

In [ ]:
import anthropic
import pandas as pd
import json
import re
import io
from datetime import datetime

client = anthropic.Anthropic()
DATASETS = {}  # nombre → DataFrame
print("Cliente y entorno listos")

## 1. Dataset de ventas de ejemplo

In [ ]:
CSV_VENTAS = """fecha,producto,categoria,importe,region,vendedor
2026-01-05,Producto A,Software,1200,Norte,Ana García
2026-01-12,Producto B,Hardware,450,Sur,Carlos López
2026-01-18,Producto A,Software,1800,Norte,Ana García
2026-01-25,Producto C,Servicios,3200,Este,María Ruiz
2026-02-03,Producto A,Software,950,Norte,Pedro Martín
2026-02-08,Producto B,Hardware,380,Sur,Carlos López
2026-02-14,Producto C,Servicios,2800,Este,María Ruiz
2026-02-20,Producto A,Software,1650,Norte,Ana García
2026-02-27,Producto D,Software,5200,Norte,Ana García
2026-03-05,Producto A,Software,1100,Norte,Pedro Martín
2026-03-10,Producto B,Hardware,520,Sur,Carlos López
2026-03-15,Producto C,Servicios,4100,Este,María Ruiz
2026-03-22,Producto D,Software,4800,Norte,Ana García
2026-03-28,Producto A,Software,900,Norte,Pedro Martín
2026-03-29,Producto E,Hardware,12000,Norte,Ana García"""

df_ventas = pd.read_csv(io.StringIO(CSV_VENTAS))
df_ventas["fecha"] = pd.to_datetime(df_ventas["fecha"])
DATASETS["ventas"] = df_ventas

print(f"Dataset 'ventas' cargado: {df_ventas.shape}")
df_ventas.head()

## 2. Definir herramientas para Claude

In [ ]:
HERRAMIENTAS = [
    {
        "name": "describir_dataset",
        "description": "Muestra estructura del dataset: shape, columnas, tipos, estadísticas básicas",
        "input_schema": {
            "type": "object",
            "properties": {"nombre": {"type": "string"}},
            "required": ["nombre"]
        }
    },
    {
        "name": "ejecutar_pandas",
        "description": "Ejecuta código Pandas sobre el dataset. Usa 'df' para el dataset y 'resultado' para el output.",
        "input_schema": {
            "type": "object",
            "properties": {
                "nombre": {"type": "string"},
                "codigo": {"type": "string", "description": "Código Pandas. Asigna resultado a 'resultado'."}
            },
            "required": ["nombre", "codigo"]
        }
    },
    {
        "name": "calcular_kpis",
        "description": "Calcula KPIs de una columna numérica, opcionalmente agrupando por otra columna",
        "input_schema": {
            "type": "object",
            "properties": {
                "nombre": {"type": "string"},
                "columna": {"type": "string"},
                "kpis": {
                    "type": "array",
                    "items": {"type": "string", "enum": ["suma", "media", "max", "min", "distribucion"]}
                },
                "agrupar_por": {"type": "string"}
            },
            "required": ["nombre", "columna", "kpis"]
        }
    },
    {
        "name": "detectar_anomalias",
        "description": "Detecta outliers en una columna numérica usando el método IQR",
        "input_schema": {
            "type": "object",
            "properties": {
                "nombre": {"type": "string"},
                "columna": {"type": "string"}
            },
            "required": ["nombre", "columna"]
        }
    }
]

print(f"{len(HERRAMIENTAS)} herramientas definidas")

## 3. Ejecución segura de código Pandas

In [ ]:
def ejecutar_seguro(df: pd.DataFrame, codigo: str) -> dict:
    """Ejecuta código Pandas bloqueando operaciones peligrosas."""
    PROHIBIDOS = [r'\bopen\b', r'\bos\b', r'\bsubprocess\b', r'\beval\b', r'\bexec\b']
    for pat in PROHIBIDOS:
        if re.search(pat, codigo):
            return {"error": f"Operación bloqueada por seguridad: {pat}"}
    
    entorno = {
        "df": df.copy(), "pd": pd, "resultado": None,
        "__builtins__": {"len": len, "range": range, "list": list, "dict": dict,
                         "str": str, "int": int, "float": float, "round": round,
                         "sum": sum, "min": min, "max": max, "sorted": sorted}
    }
    try:
        exec(codigo, entorno)
        r = entorno.get("resultado")
        if isinstance(r, pd.DataFrame):
            return {"tipo": "dataframe", "datos": r.head(10).to_dict(orient="records"), "filas": len(r)}
        elif isinstance(r, pd.Series):
            return {"tipo": "serie", "datos": r.to_dict()}
        elif r is not None:
            return {"tipo": "valor", "datos": r}
        return {"tipo": "ok", "mensaje": "Sin valor de retorno"}
    except Exception as e:
        return {"error": str(e)}

def ejecutar_herramienta(nombre: str, params: dict) -> str:
    if nombre == "describir_dataset":
        ds = params["nombre"]
        if ds not in DATASETS: return json.dumps({"error": f"Dataset '{ds}' no encontrado"})
        df = DATASETS[ds]
        return json.dumps({
            "shape": df.shape, "columnas": list(df.columns),
            "tipos": df.dtypes.astype(str).to_dict(),
            "nulos": df.isnull().sum().to_dict(),
            "estadisticas": df.describe().to_dict()
        }, default=str)
    
    elif nombre == "ejecutar_pandas":
        ds = params["nombre"]
        if ds not in DATASETS: return json.dumps({"error": f"Dataset '{ds}' no encontrado"})
        return json.dumps(ejecutar_seguro(DATASETS[ds], params["codigo"]), default=str)
    
    elif nombre == "calcular_kpis":
        ds, col = params["nombre"], params["columna"]
        if ds not in DATASETS: return json.dumps({"error": f"Dataset '{ds}' no encontrado"})
        df = DATASETS[ds]
        if col not in df.columns: return json.dumps({"error": f"Columna '{col}' no existe"})
        agrupar = params.get("agrupar_por")
        serie = df.groupby(agrupar)[col] if agrupar and agrupar in df.columns else df[col]
        res = {}
        for kpi in params.get("kpis", []):
            if kpi == "suma": res["suma"] = float(serie.sum()) if not agrupar else serie.sum().to_dict()
            elif kpi == "media": res["media"] = float(serie.mean()) if not agrupar else serie.mean().to_dict()
            elif kpi == "max": res["max"] = float(serie.max()) if not agrupar else serie.max().to_dict()
            elif kpi == "min": res["min"] = float(serie.min()) if not agrupar else serie.min().to_dict()
            elif kpi == "distribucion": res["distribucion"] = df[col].value_counts().head(10).to_dict()
        return json.dumps(res, default=str)
    
    elif nombre == "detectar_anomalias":
        ds, col = params["nombre"], params["columna"]
        if ds not in DATASETS: return json.dumps({"error": f"Dataset '{ds}' no encontrado"})
        df = DATASETS[ds]
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
        return json.dumps({
            "total_outliers": len(outliers), "pct": round(len(outliers)/len(df)*100, 2),
            "limite_inf": round(float(Q1 - 1.5*IQR), 2), "limite_sup": round(float(Q3 + 1.5*IQR), 2),
            "valores_outlier": outliers[col].tolist()
        })
    
    return json.dumps({"error": f"Herramienta '{nombre}' desconocida"})

# Probar herramientas directamente
print("Test: calcular KPIs de importe por región")
res = ejecutar_herramienta("calcular_kpis", {"nombre": "ventas", "columna": "importe", 
                                               "kpis": ["suma", "media"], "agrupar_por": "region"})
print(json.dumps(json.loads(res), indent=2))

## 4. Bucle agéntico

In [ ]:
def analizar_con_agente(pregunta: str, max_iter: int = 6) -> dict:
    """Claude razona, llama herramientas, interpreta resultados."""
    mensajes = [{"role": "user", "content": pregunta}]
    herramientas_usadas = []
    
    for iteracion in range(max_iter):
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1500,
            system="""Eres un analista de datos experto. El usuario hace preguntas en lenguaje natural.
Usa las herramientas para obtener datos concretos antes de responder.
Siempre: 1) Explora la estructura del dataset. 2) Analiza los datos. 3) Interpreta los resultados.""",
            tools=HERRAMIENTAS,
            messages=mensajes
        )
        
        mensajes.append({"role": "assistant", "content": resp.content})
        
        if resp.stop_reason == "end_turn":
            texto = next((b.text for b in resp.content if hasattr(b, "text")), "")
            return {"respuesta": texto, "iteraciones": iteracion+1, "herramientas": herramientas_usadas}
        
        if resp.stop_reason == "tool_use":
            resultados = []
            for bloque in resp.content:
                if bloque.type == "tool_use":
                    herramientas_usadas.append(bloque.name)
                    print(f"  [{iteracion+1}] → {bloque.name}({list(bloque.input.keys())})")
                    resultado = ejecutar_herramienta(bloque.name, bloque.input)
                    resultados.append({"type": "tool_result", "tool_use_id": bloque.id, "content": resultado})
            mensajes.append({"role": "user", "content": resultados})
    
    return {"respuesta": "Límite de iteraciones alcanzado.", "iteraciones": max_iter, "herramientas": herramientas_usadas}

print("Agente listo")

In [ ]:
# Consulta 1: análisis de ventas por región y vendedor
print("CONSULTA 1: Análisis de ventas")
print("=" * 55)
resultado = analizar_con_agente(
    "Analiza el dataset 'ventas'. ¿Qué región y vendedor generan más ingresos? "
    "¿Hay alguna anomalía o dato atípico que deba preocuparme?"
)
print(f"\nRESPUESTA ({resultado['iteraciones']} iteraciones, {len(resultado['herramientas'])} llamadas):")
print(resultado["respuesta"])

In [ ]:
# Consulta 2: tendencia temporal
print("CONSULTA 2: Tendencia temporal")
print("=" * 55)
resultado2 = analizar_con_agente(
    "Del dataset 'ventas', ¿cuál es la tendencia mensual de ingresos? "
    "¿En qué mes se vendió más y por qué crees que fue así?"
)
print(f"\nRESPUESTA ({resultado2['iteraciones']} iteraciones):")
print(resultado2["respuesta"])

## 5. Métricas del agente

In [ ]:
from collections import Counter

todas_herramientas = resultado["herramientas"] + resultado2["herramientas"]
conteo = Counter(todas_herramientas)

print("USO DE HERRAMIENTAS (ambas consultas)")
print("=" * 40)
for herramienta, n in conteo.most_common():
    print(f"  {herramienta:25} {n}x")

print(f"\nTotal iteraciones: {resultado['iteraciones'] + resultado2['iteraciones']}")
print(f"Total llamadas API: {len(todas_herramientas)}")
print("\n💡 El agente determina automáticamente qué herramientas usar y en qué orden.")

## Resumen

| Componente | Función |
|---|---|
| Tool use nativo | Claude decide qué herramientas usar y cuándo |
| Sandbox de ejecución | Código Pandas sin acceso a filesystem/red |
| Bucle agéntico | Hasta 6 iteraciones: razona → actúa → observa |
| Modelo | Sonnet para razonamiento complejo sobre datos |

**Extensiones naturales:**
- Añadir herramienta `generar_grafico` con matplotlib → base64
- Soporte para múltiples datasets con `JOIN` entre ellos
- Memoria de sesión para preguntas de seguimiento